Imports

In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
import json, math, editdistance

Vocab + model classes

In [25]:
with open('../src/vocab.json') as f:
    vocab = json.load(f)
stoi = vocab['stoi']
itos = {int(k): v for k, v in vocab['itos'].items()}
PAD, BOS, EOS = 0, 1, 2
vocab_size = 39
device = 'cuda' if torch.cuda.is_available() else 'cpu'

class CNNEncoder(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d((2, 1)),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d((2, 1)),
            nn.Conv2d(256, d_model, 3, padding=1), nn.BatchNorm2d(d_model), nn.ReLU(), nn.MaxPool2d((2, 1)),
        )
    def forward(self, x):
        x = self.conv(x)
        x = x.squeeze(2)
        return x.permute(0, 2, 1)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.h = n_heads
        self.dk = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
    def forward(self, q_in, kv_in, mask=None):
        B, Tq, _ = q_in.shape
        Tk = kv_in.shape[1]
        Q = self.q_proj(q_in).view(B, Tq, self.h, self.dk).transpose(1, 2)
        K = self.k_proj(kv_in).view(B, Tk, self.h, self.dk).transpose(1, 2)
        V = self.v_proj(kv_in).view(B, Tk, self.h, self.dk).transpose(1, 2)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.dk)
        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))
        attn = scores.softmax(dim=-1)
        out = attn @ V
        out = out.transpose(1, 2).contiguous().view(B, Tq, -1)
        return self.out_proj(out)

def causal_mask(T, device):
    return torch.triu(torch.ones(T, T, dtype=torch.bool, device=device), diagonal=1)

class FeedForward(nn.Module):
    def __init__(self, d_model, ff_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, ff_dim), nn.GELU(), nn.Linear(ff_dim, d_model))
    def forward(self, x):
        return self.net(x)

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, ff_dim):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.cross_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = FeedForward(d_model, ff_dim)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
    def forward(self, x, memory, self_mask):
        normed = self.norm1(x)
        x = x + self.self_attn(normed, normed, mask=self_mask)
        x = x + self.cross_attn(self.norm2(x), memory, mask=None)
        x = x + self.ffn(self.norm3(x))
        return x

class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_heads=8, n_layers=3, ff_dim=1024, max_len=50):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, ff_dim) for _ in range(n_layers)])
        self.norm_out = nn.LayerNorm(d_model)
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.d_model = d_model
    def forward(self, tgt_in, memory):
        B, T = tgt_in.shape
        x = self.embed(tgt_in) * math.sqrt(self.d_model)
        x = self.pos_enc(x)
        mask = causal_mask(T, tgt_in.device)
        for layer in self.layers:
            x = layer(x, memory, mask)
        x = self.norm_out(x)
        return self.fc_out(x)

class OCRModel(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_heads=8, n_layers=3, ff_dim=1024):
        super().__init__()
        self.encoder = CNNEncoder(d_model)
        self.decoder = Decoder(vocab_size, d_model, n_heads, n_layers, ff_dim)

    def forward(self, imgs, tgt_in):
        memory = self.encoder(imgs)
        return self.decoder(tgt_in, memory)

Load the trained weights

In [26]:
model = OCRModel(vocab_size=vocab_size).to(device)
model.load_state_dict(torch.load('../checkpoints/model_50k_v2_best.pt', map_location=device))
model.eval()

OCRModel(
  (encoder): CNNEncoder(
    (conv): Sequential(
      (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU()
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (5): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (6): ReLU()
      (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (8): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (9): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (10): ReLU()
      (11): MaxPool2d(kernel_size=(2, 1), stride=(2, 1), padding=0, dilation=1, ceil_mode=False)
      (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 

Autoregression

In [27]:
@torch.no_grad()
def generate(model, img, max_len=25):
    model.eval()
    img = img.unsqueeze(0).to(device)          # add batch dim: [1, 1, 32, 128]
    memory = model.encoder(img)                 # [1, 32, 256]

    seq = [BOS]
    for _ in range(max_len):
        tgt = torch.tensor([seq], device=device)
        logits = model.decoder(tgt, memory)      # [1, len(seq), vocab_size]
        next_id = logits[0, -1].argmax().item()   # only look at LAST position
        if next_id == EOS:
            break
        seq.append(next_id)

    return ''.join(itos[i] for i in seq[1:])       # drop BOS

Testing on img 0

In [28]:
transform = T.Compose([
    T.Resize((32, 128)), T.Grayscale(), T.ToTensor(), T.Normalize([0.5], [0.5]),
])

img = transform(Image.open('../data/synth/train/00000.png').convert('RGB'))
pred = generate(model, img)
print(pred)

0t2be1yq


Metrics

In [29]:
def normalize(s):
    return ''.join(c for c in s.lower() if c.isalnum())

def word_accuracy(preds, labels):
    correct = sum(normalize(p) == normalize(l) for p, l in zip(preds, labels))
    return correct / len(labels)

def cer(preds, labels):
    total_dist = sum(editdistance.eval(normalize(p), normalize(l)) for p, l in zip(preds, labels))
    total_chars = sum(len(normalize(l)) for l in labels)
    return total_dist / total_chars

Full eval on val split

In [30]:
eval_paths, eval_words = [], []
with open("../data/synth/eval_fresh/labels.txt") as f:
    for line in f:
        fname, word = line.strip().split("\t")
        eval_paths.append(f"../data/synth/eval_fresh/{fname}")
        eval_words.append(word)

print(len(eval_paths))

preds = []
for p in eval_paths:
    img = transform(Image.open(p).convert('RGB'))
    preds.append(generate(model, img))

acc = word_accuracy(preds, eval_words)
error_rate = cer(preds, eval_words)
print(f"word accuracy: {acc:.4f}")
print(f"CER: {error_rate:.4f}")

1000
word accuracy: 0.9640
CER: 0.0136


In [31]:
mistakes = [(l, p) for l, p in zip(eval_words, preds) if normalize(l) != normalize(p)]
print(len(mistakes), "mistakes out of", len(eval_words))
for l, p in mistakes[:15]:
    print(f"true: {l:15s}  pred: {p}")

36 mistakes out of 1000
true: hsq8tq6wtr       pred: hsq6wtq6wtr
true: l6bfldlxo        pred: l6bflxo
true: ea5cg4ejb        pred: ea5cg4ea5b
true: 5uksb5gx         pred: 5gb5uksb
true: 051rvh06x        pred: 061rvh06x
true: q6pcqo           pred: qcq6pcqo
true: w6ysms3f1k       pred: w6ys3f1k
true: 8mkmm2n3         pred: 8mmkm2n3
true: ufmuq            pred: uqfmu
true: yl2h63ve7x       pred: yp72h63ve7x
true: euxrpxax         pred: euxaxrpx
true: 4docgicxs        pred: 4docxs
true: px4x71fj         pred: px71fj
true: fufza            pred: fza
true: w8boxcw1ii       pred: w1i3boxcw1i


Dictionary-words VS Random words test

In [32]:

dictionary_words = ["stop", "hello", "world", "notification", "at", "the", "and",
                     "recognition", "transformer", "python", "image", "attention",
                     "network", "training", "model", "vision", "encoder", "decoder"]
dict_pairs = [(l, p) for l, p in zip(eval_words, preds) if l in dictionary_words]
rand_pairs = [(l, p) for l, p in zip(eval_words, preds) if l not in dictionary_words]

print("dict word accuracy:", word_accuracy([p for l,p in dict_pairs], [l for l,p in dict_pairs]))
print("random string accuracy:", word_accuracy([p for l,p in rand_pairs], [l for l,p in rand_pairs]))

dict word accuracy: 0.9985835694050992
random string accuracy: 0.8809523809523809


Repetition penalised generate

In [5]:
@torch.no_grad()
def generate_v2(model, img, max_len=25, rep_penalty=1.3, rep_window=2):
    model.eval()
    img = img.unsqueeze(0).to(device)
    memory = model.encoder(img)

    seq = [BOS]
    for _ in range(max_len):
        tgt = torch.tensor([seq], device=device)
        logits = model.decoder(tgt, memory)[0, -1]   # [vocab_size]

        # penalize characters seen in the last `rep_window` positions
        recent = seq[-rep_window:]
        for tok in set(recent):
            if tok not in (PAD, BOS, EOS):
                logits[tok] = logits[tok] / rep_penalty if logits[tok] > 0 else logits[tok] * rep_penalty

        next_id = logits.argmax().item()
        if next_id == EOS:
            break
        seq.append(next_id)

    return ''.join(itos[i] for i in seq[1:])

In [6]:
eval_paths, eval_words = [], []
with open("../data/synth/eval_fresh/labels.txt") as f:
    for line in f:
        fname, word = line.strip().split("\t")
        eval_paths.append(f"../data/synth/eval_fresh/{fname}")
        eval_words.append(word)

In [ ]:
from tqdm import tqdm
transform = T.Compose([
    T.Resize((32, 128)), T.Grayscale(), T.ToTensor(), T.Normalize([0.5], [0.5]),
])
preds_v2 = []
for p in tqdm(eval_paths):
    img = transform(Image.open(p).convert('RGB'))
    preds_v2.append(generate_v2(model, img))


100%|██████████| 1000/1000 [00:34<00:00, 29.23it/s]


NameError: name 'word_accuracy' is not defined

In [15]:

acc = word_accuracy(preds_v2, eval_words)
error_rate = cer(preds_v2, eval_words)
print(f"word accuracy: {acc:.4f}")
print(f"CER: {error_rate:.4f}")

dictionary_words = ["stop", "hello", "world", "notification", "at", "the", "and",
                     "recognition", "transformer", "python", "image", "attention",
                     "network", "training", "model", "vision", "encoder", "decoder"]


dict_pairs = [(l, p) for l, p in zip(eval_words, preds_v2) if l in dictionary_words]
rand_pairs = [(l, p) for l, p in zip(eval_words, preds_v2) if l not in dictionary_words]
print("dict accuracy:", word_accuracy([p for l,p in dict_pairs], [l for l,p in dict_pairs]))
print("random accuracy:", word_accuracy([p for l,p in rand_pairs], [l for l,p in rand_pairs]))

word accuracy: 0.9290
CER: 0.0280
dict accuracy: 0.9660056657223796
random accuracy: 0.8401360544217688


In [19]:
for l, p in [("nomw5","?"), ("transformer","?"), ("t7fhh2w","?"), ("axkj4f","?")]:
    idx = eval_words.index(l) if l in eval_words else None
    if idx is not None:
        print(l, "->", preds_v2[idx])

nomw5 -> nomonw5
transformer -> transformer
t7fhh2w -> t7fhh2w
axkj4f -> axkkj4f


In [20]:
mistakes = [(l, p) for l, p in zip(eval_words, preds_v2) if normalize(l) != normalize(p)]
print(len(mistakes), "mistakes out of", len(eval_words))
for l, p in mistakes[:15]:
    print(f"true: {l:15s}  pred: {p}")

71 mistakes out of 1000
true: nomw5            pred: nomonw5
true: ykmkwpg          pred: ykwpg
true: l6bfldlxo        pred: l6bflxo
true: transformer      pred: transformermer
true: ea5cg4ejb        pred: ea5cg4ea5b
true: transformer      pred: transformermer
true: hello            pred: helo
true: 5uksb5gx         pred: 5gb5uksb5
true: qpp6             pred: qp6
true: 051rvh06x        pred: 061rvh06x
true: transformer      pred: transformermer
true: w6ysms3f1k       pred: w6ysms3flk
true: uxxkg1ve         pred: uxkg1ve
true: 8mkmm2n3         pred: 8m2nkm2n3
true: axkj4f           pred: axkkj4f


In [21]:
indices = [i for i, w in enumerate(eval_words) if w == "transformer"]
print(len(indices), indices)

for i in indices:
    print(i, "true:", eval_words[i], " pred:", preds_v2[i])

46 [19, 39, 68, 69, 103, 130, 226, 250, 251, 255, 275, 278, 289, 290, 324, 327, 331, 352, 374, 391, 413, 423, 457, 467, 530, 535, 547, 568, 572, 592, 606, 607, 672, 742, 757, 771, 788, 895, 901, 928, 947, 969, 978, 984, 997, 998]
19 true: transformer  pred: transformer
39 true: transformer  pred: transformermer
68 true: transformer  pred: transformermer
69 true: transformer  pred: transformer
103 true: transformer  pred: transformermer
130 true: transformer  pred: transformer
226 true: transformer  pred: trmeformeran
250 true: transformer  pred: transformermer
251 true: transformer  pred: transformermer
255 true: transformer  pred: transformer
275 true: transformer  pred: transformer
278 true: transformer  pred: transformermer
289 true: transformer  pred: transfomer
290 true: transformer  pred: transformer
324 true: transformer  pred: transformermer
327 true: transformer  pred: transformer
331 true: transformer  pred: transformer
352 true: transformer  pred: transformermer
374 true: tr

In [22]:
transformer_results = [(i, preds_v2[i]) for i in indices]
correct = [i for i, p in transformer_results if normalize(p) == 'transformer']
wrong = [i for i, p in transformer_results if normalize(p) != 'transformer']
print(f"{len(correct)} correct, {len(wrong)} wrong out of {len(indices)}")

23 correct, 23 wrong out of 46


In [23]:
from collections import defaultdict

by_length = defaultdict(lambda: [0, 0])   # length -> [correct, total]
for l, p in zip(eval_words, preds_v2):
    length = len(l)
    by_length[length][1] += 1
    if normalize(l) == normalize(p):
        by_length[length][0] += 1

for length in sorted(by_length):
    correct, total = by_length[length]
    print(f"length {length}: {correct}/{total} = {correct/total:.2%}")

length 2: 72/72 = 100.00%
length 3: 104/106 = 98.11%
length 4: 76/77 = 98.70%
length 5: 181/189 = 95.77%
length 6: 94/95 = 98.95%
length 7: 145/150 = 96.67%
length 8: 63/71 = 88.73%
length 9: 72/82 = 87.80%
length 10: 19/32 = 59.38%
length 11: 59/82 = 71.95%
length 12: 44/44 = 100.00%
